In [ ]:
# Importing Libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.metrics import classification_report, accuracy_score

# Loading Dataset

In [ ]:
df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
df.head()

# Checking dataset correctly loaded or not?

In [ ]:
df.info()

>'Analysis' : Dataset correctly loaded

# Checking Duplicate rows or missing value

In [ ]:
df.duplicated().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.isna().sum()/df.shape[0]*100

# target distribution

In [ ]:
plt.figure(figsize = (5,5))
df['target'].value_counts().plot.pie(autopct = '%1.1f%%',colors = ['skyblue', 'salmon'], labels = ['Not Disaster', 'Disaster'])
plt.title('Target Distribution')
plt.show()

# Data cleaning

In [ ]:
def clean_txt(text):
    text = text.lower()
    text = re.sub(r'http\S+|www.\s+','', text)
    text = re.sub(r'@\w+|#','', text)
    text = re.sub(r'[^a-zA-Z\s]','', text)
    text = re.sub(r'\s+','', text).strip()
    return text
    

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_word = set(stopwords.words('english'))

In [ ]:
def preprocess(text):
    text = clean_txt(text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_word]
    return ' '.join(words)

In [ ]:
df['clean_text'] = df['text'].apply(preprocess)

# Feature Extraction using TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
vectorizer = TfidfVectorizer(max_features = 5000)
X = vectorizer.fit_transform(df['clean_text']).toarray()
y = df['target']

# Train-Test-Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X,y, test_size = 0.2, random_state = 52)

# Model Training

In [ ]:
# LOGISTIC REGRESSION
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression(max_iter = 1000)
lr.fit(Xtrain, ytrain)
y_pred_lr = lr.predict(Xtest)
print('Logistic Regression : ', accuracy_score(ytest, y_pred_lr))

# Prediction on Test Data

In [ ]:
test_df  = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
test_df.head()

In [ ]:
# preprocessing test data - 
test_df['Clean_text']= test_df['text'].apply(preprocess)
X_test_final = vectorizer.transform(test_df['Clean_text']).toarray()
test_pred = lr.predict(X_test_final)
#preparing submission file
submission = pd.DataFrame({'id': test_df['id'], 'target': test_pred})
submission.to_csv('submission.csv', index = False)